# Lesson 3: Chatbot Example

In this lesson, you will familiarize yourself with the chatbot example you will work on during this course. The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.

## Import Libraries

In [1]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

## Tool Functions

In [2]:
PAPER_DIR = "papers"

The first tool searches for relevant arXiv papers based on a topic and stores the papers' info in a JSON file (title, authors, summary, paper url and the publication date). The JSON files are organized by topics in the `papers` directory. The tool does not download the papers.  

In [3]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    
    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info  
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids

In [5]:
search_papers("diffusion")

Results are saved in: papers/diffusion/papers_info.json


['1701.00257v2',
 '1602.06439v1',
 '2001.00852v2',
 'math/0204289v1',
 '1611.06202v2']

The second tool looks for information about a specific paper across all topic directories inside the `papers` directory.

In [6]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

In [7]:
extract_info('1310.7911v2')

'{\n  "title": "Compact manifolds with computable boundaries",\n  "authors": [\n    "Zvonko Iljazovic"\n  ],\n  "summary": "We investigate conditions under which a co-computably enumerable closed set\\nin a computable metric space is computable and prove that in each locally\\ncomputable computable metric space each co-computably enumerable compact\\nmanifold with computable boundary is computable. In fact, we examine the notion\\nof a semi-computable compact set and we prove a more general result: in any\\ncomputable metric space each semi-computable compact manifold with computable\\nboundary is computable. In particular, each semi-computable compact\\n(boundaryless) manifold is computable.",\n  "pdf_url": "http://arxiv.org/pdf/1310.7911v2",\n  "published": "2013-10-29"\n}'

In [8]:
extract_info('1701.00257v2')

'{\n  "title": "Analyzing PFG anisotropic anomalous diffusions by instantaneous signal attenuation method",\n  "authors": [\n    "Guoxing Lin"\n  ],\n  "summary": "Anomalous diffusion has been investigated in many systems. Pulsed field\\ngradient (PFG) anomalous diffusion is much more complicated than PFG normal\\ndiffusion. There have been many theoretical and experimental studies for PFG\\nisotropic anomalous diffusion, but there are very few theoretical treatments\\nreported for anisotropic anomalous diffusion. Currently, there is not a general\\nPFG signal attenuation expression, which includes the finite gradient pulse\\neffect and can treat all three types of anisotropic fractional diffusions:\\ngeneral fractional diffusion, time fractional diffusion, and space-fractional\\ndiffusion. In this paper, the recently developed instantaneous signal\\nattenuation (ISA) method was applied to obtain PFG signal attenuation\\nexpression for free and restricted anisotropic anomalous diffusio

In [15]:
extract_info('2006.11239')

"There's no saved information related to paper 2006.11239."

In [16]:
search_papers('Denoising Diffusion Probabilistic Models')

Results are saved in: papers/denoising_diffusion_probabilistic_models/papers_info.json


['2307.15988v1',
 '2410.11511v1',
 '2312.01682v1',
 '2412.01371v1',
 '2006.11239v2']

In [17]:
extract_info('2006.11239v2')

'{\n  "title": "Denoising Diffusion Probabilistic Models",\n  "authors": [\n    "Jonathan Ho",\n    "Ajay Jain",\n    "Pieter Abbeel"\n  ],\n  "summary": "We present high quality image synthesis results using diffusion probabilistic\\nmodels, a class of latent variable models inspired by considerations from\\nnonequilibrium thermodynamics. Our best results are obtained by training on a\\nweighted variational bound designed according to a novel connection between\\ndiffusion probabilistic models and denoising score matching with Langevin\\ndynamics, and our models naturally admit a progressive lossy decompression\\nscheme that can be interpreted as a generalization of autoregressive decoding.\\nOn the unconditional CIFAR10 dataset, we obtain an Inception score of 9.46 and\\na state-of-the-art FID score of 3.17. On 256x256 LSUN, we obtain sample quality\\nsimilar to ProgressiveGAN. Our implementation is available at\\nhttps://github.com/hojonathanho/diffusion",\n  "pdf_url": "http://arxi

In [21]:
json.loads(extract_info('2006.11239v2')).keys()

dict_keys(['title', 'authors', 'summary', 'pdf_url', 'published'])

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [9]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

This code handles tool mapping and execution.

In [10]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [11]:
load_dotenv() 
client = anthropic.Anthropic()

### Query Processing

In [26]:
def process_query(query):
    
    messages = [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages)
    
    process_query = True
    while process_query:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query = False
            
            elif content.type == 'tool_use':
                print('content', content)
                
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({"role": "user", 
                                  "content": [
                                      {
                                          "type": "tool_result",
                                          "tool_use_id": tool_id,
                                          "content": result
                                      }
                                  ]
                                })
                response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False

### Chat Loop

In [13]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the chatbot. Here's an example query: 

- Search for 2 papers on "LLM interpretability"

To access the `papers` folder: 1) click on the `File` option on the top menu of the notebook and 2) click on `Open` and then 3) click on `L3`.

In [23]:
chat_loop()

Type your queries or 'quit' to exit.



Query:  hi


Hello! I'm here to help you search for and extract information about academic papers on arXiv. 

I can:
- Search for papers on a specific topic
- Extract detailed information about a specific paper using its ID

What would you like to know about? You can ask me to search for papers on a particular topic, or if you have a specific arXiv paper ID, I can get detailed information about that paper.





Query:  Denoising Diffusion Probabilistic Models


I'll help you find information about Denoising Diffusion Probabilistic Models. This is a topic in machine learning related to generative models. Let me search for relevant papers on arXiv for you.
tool_args {'topic': 'Denoising Diffusion Probabilistic Models', 'max_results': 5}
Calling tool search_papers with args {'topic': 'Denoising Diffusion Probabilistic Models', 'max_results': 5}
Results are saved in: papers/denoising_diffusion_probabilistic_models/papers_info.json
I've found several papers on Denoising Diffusion Probabilistic Models. Let me get more detailed information about each of them, starting with what appears to be one of the foundational papers (2006.11239v2):
tool_args {'paper_id': '2006.11239v2'}
Calling tool extract_info with args {'paper_id': '2006.11239v2'}
Now, let me get information about the more recent papers:
tool_args {'paper_id': '2307.15988v1'}
Calling tool extract_info with args {'paper_id': '2307.15988v1'}
tool_args {'paper_id': '2410.11511v1'}
Calling tool


Query:  Tell me more about what specific aspects I should know


I'd be happy to help you find information, but your request is quite general. To provide you with relevant information, I need to know:

1. What specific topic or subject are you interested in learning more about?
2. Are you looking for academic papers on a particular field or concept?

Once you provide more details about what aspects you're interested in, I can search for appropriate papers or information to help you.





Query:  I'm good, thank you


I'd be happy to help you with a research question or task. However, you haven't specified what information you're looking for or what topic you'd like me to search for papers about.

I can search for academic papers on arXiv on various topics, or I can extract detailed information about a specific paper if you have its ID. 

Would you like me to:
1. Search for papers on a particular topic? (Please specify the topic)
2. Extract information about a specific paper? (Please provide the paper ID)

Please let me know how I can assist you with your research needs.





Query:  quit


In [27]:
chat_loop()

Type your queries or 'quit' to exit.



Query:  hi


Hello! I'm here to help you search for and extract information about academic papers on arXiv. 

I can:
- Search for papers on a specific topic
- Extract detailed information about a specific paper using its ID

What would you like to know about? You can ask me to search for papers on a particular topic, or if you have a specific arXiv paper ID, I can get detailed information about that paper.





Query:  Denoising Diffusion Probabilistic Models


I'll help you find information about Denoising Diffusion Probabilistic Models. This is a topic in machine learning related to generative models. Let me search for relevant papers on arXiv for you.
content ToolUseBlock(id='toolu_01AfRFug47iP7vb2KYdmjpYF', input={'topic': 'Denoising Diffusion Probabilistic Models', 'max_results': 5}, name='search_papers', type='tool_use')
Calling tool search_papers with args {'topic': 'Denoising Diffusion Probabilistic Models', 'max_results': 5}
Results are saved in: papers/denoising_diffusion_probabilistic_models/papers_info.json
I've found several papers on Denoising Diffusion Probabilistic Models. Let me get more detailed information about each of them, starting with what appears to be one of the foundational papers (2006.11239v2):
content ToolUseBlock(id='toolu_019qRYZZD6PRcLhLfNV2oDwR', input={'paper_id': '2006.11239v2'}, name='extract_info', type='tool_use')
Calling tool extract_info with args {'paper_id': '2006.11239v2'}
Now, let me get informatio


Query:  quit


<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b> To Access the <code>requirements.txt</code> file or the <code>papers</code> folder: </b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em> and finally 3) click on <em>"L3"</em>.
</div>

In the next lessons, you will take out the tool definitions to wrap them in an MCP server. Then you will create an MCP client inside the chatbot to make the chatbot MCP compatible.  

## Resources

[Guide on how to implement tool use](https://docs.anthropic.com/en/docs/build-with-claude/tool-use/overview#how-to-implement-tool-use)

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">


<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

</div>